# Compressor v3 Digital Twin - Telemetry + Labels Generation


In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from tqdm import tqdm


In [2]:

class CompressorTwin:

    def __init__(self, asset_id, failure_scenario):

        self.asset_id = asset_id
        self.failure_scenario = failure_scenario

        self.health = 100.0
        self.runtime_hours = 0

        self.bearing_wear = 0
        self.oil_degradation = 0
        self.air_leakage = 0

    def update_health(self):

        wear = np.random.uniform(0.00015, 0.0008)

        if self.failure_scenario == "Bearing Failure":
            self.bearing_wear += wear * 5
            self.health -= wear * 1.8

        elif self.failure_scenario == "Oil Degradation":
            self.oil_degradation += wear * 5
            self.health -= wear * 1.5

        elif self.failure_scenario == "Air Leakage":
            self.air_leakage += wear * 5
            self.health -= wear * 1.3

        else:
            self.health -= wear * 0.3

        self.health = max(self.health, 0)
        self.runtime_hours += 1/60

    def get_rul(self):
        return round((self.health / 100) * 220, 2)

    def get_failure_next_30_days(self):
        return int(self.get_rul() <= 30)

    def generate_telemetry(self):

        self.update_health()

        inlet_pressure = 2.5 + np.random.normal(0,0.05)

        outlet_pressure = (
            12 - self.air_leakage * 0.5 +
            np.random.normal(0,0.1)
        )

        compression_ratio = outlet_pressure / inlet_pressure

        air_flow_rate = (
            500 - self.air_leakage * 8 +
            np.random.normal(0,3)
        )

        bearing_temp = (
            65 + self.bearing_wear * 10 +
            np.random.normal(0,1)
        )

        oil_temp = (
            50 + self.oil_degradation * 8 +
            np.random.normal(0,1)
        )

        oil_pressure = (
            4 - self.oil_degradation * 0.3 +
            np.random.normal(0,0.05)
        )

        vibration = (
            1.5 + self.bearing_wear * 2 +
            np.random.normal(0,0.05)
        )

        motor_current = (
            35 + self.bearing_wear * 4 +
            np.random.normal(0,0.2)
        )

        motor_voltage = 440 + np.random.normal(0,2)

        power_consumption = (
            motor_current * motor_voltage / 1000
        )

        telemetry = {
            "asset_id": self.asset_id,
            "inlet_pressure": round(inlet_pressure,2),
            "outlet_pressure": round(outlet_pressure,2),
            "compression_ratio": round(compression_ratio,2),
            "air_flow_rate": round(air_flow_rate,2),
            "bearing_temp": round(bearing_temp,2),
            "oil_temp": round(oil_temp,2),
            "oil_pressure": round(oil_pressure,2),
            "vibration": round(vibration,2),
            "motor_current": round(motor_current,2),
            "motor_voltage": round(motor_voltage,2),
            "power_consumption": round(power_consumption,2)
        }

        labels = {
            "asset_id": self.asset_id,
            "failure_mode": self.failure_scenario,
            "rul_days": self.get_rul(),
            "failure_next_30_days": self.get_failure_next_30_days()
        }

        return telemetry, labels


In [3]:

compressors = [
    CompressorTwin("C-201","Bearing Failure"),
    CompressorTwin("C-202","Oil Degradation"),
    CompressorTwin("C-203","Air Leakage"),
    CompressorTwin("C-204","Bearing Failure"),
    CompressorTwin("C-205","Healthy")
]


In [4]:

telemetry_records = []
label_records = []

minutes = 90 * 24 * 60
start_time = datetime.now()

for minute in tqdm(range(minutes)):

    current_time = start_time + timedelta(minutes=minute)

    for comp in compressors:

        telemetry, labels = comp.generate_telemetry()

        telemetry["timestamp"] = current_time
        labels["timestamp"] = current_time

        telemetry_records.append(telemetry)
        label_records.append(labels)

telemetry_df = pd.DataFrame(telemetry_records)
labels_df = pd.DataFrame(label_records)

print("Telemetry Shape:", telemetry_df.shape)
print("Labels Shape:", labels_df.shape)

telemetry_df.to_csv("compressor_telemetry.csv", index=False)
labels_df.to_csv("compressor_labels.csv", index=False)

telemetry_df.head()


100%|██████████| 129600/129600 [00:17<00:00, 7565.75it/s]


Telemetry Shape: (648000, 13)
Labels Shape: (648000, 5)


,asset_id,inlet_pressure,outlet_pressure,compression_ratio,air_flow_rate,bearing_temp,oil_temp,oil_pressure,vibration,motor_current,motor_voltage,power_consumption,timestamp
0,C-201,2.49,12.10,4.86,503.26,65.01,49.72,3.99,1.53,34.95,440.63,15.40,2026-06-10 10:39:14.144194
1,C-202,2.44,11.94,4.89,503.78,65.78,49.18,4.02,1.59,35.14,436.79,15.35,2026-06-10 10:39:14.144194
2,C-203,2.41,11.97,4.96,498.75,65.32,50.58,3.98,1.54,34.76,441.54,15.35,2026-06-10 10:39:14.144194
3,C-204,2.54,11.93,4.70,499.55,64.57,49.57,4.00,1.43,34.84,439.20,15.30,2026-06-10 10:39:14.144194
4,C-205,2.52,12.07,4.79,497.36,65.06,48.20,3.98,1.46,35.05,440.85,15.45,2026-06-10 10:39:14.144194
